In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ketharnathr/inpatient/Train_Inpatientdata-1542865627584.csv


# Inpatient Claims Data Preprocessing

## Objective

The objective of this notebook is to preprocess the raw inpatient claims
dataset before it is integrated with the outpatient and beneficiary datasets.

This stage focuses ONLY on data preprocessing.

### Preprocessing goals

- Understand the structure of the raw dataset
- Inspect columns and data types
- Identify missing values
- Identify duplicate records
- Validate identifiers
- Convert date columns into appropriate datetime format
- Check date consistency
- Validate numerical values
- Standardize categorical/code columns
- Handle missing values appropriately
- Produce a clean inpatient dataset

### Important

Feature engineering and provider-level aggregation are NOT performed in
this preprocessing stage.

Those operations will be performed later on the unified dataset after
the inpatient, outpatient, and beneficiary datasets are integrated.

## 1. Load Raw Inpatient Dataset

The first step is to load the original inpatient claims CSV file.

At this stage, the data is kept as close as possible to its original
representation so that the original structure can be inspected before
any transformations are applied.

In [2]:
# ============================================================
# STEP 1: LOAD RAW INPATIENT DATA
# ============================================================

file_path = "/kaggle/input/datasets/ketharnathr/inpatient/Train_Inpatientdata-1542865627584.csv"

df = pd.read_csv(file_path)

print("=" * 80)
print("RAW INPATIENT DATA LOADED")
print("=" * 80)

print(f"Number of rows    : {df.shape[0]:,}")
print(f"Number of columns : {df.shape[1]:,}")
print(f"Total cells       : {df.size:,}")

print("=" * 80)

RAW INPATIENT DATA LOADED
Number of rows    : 40,474
Number of columns : 30
Total cells       : 1,214,220


## 2. Initial Dataset Preview

The first few records are inspected to understand:

- How the columns are populated
- The format of identifiers
- Date representation
- Physician fields
- Diagnosis codes
- Procedure codes
- Financial values
- Presence of missing values

In [3]:
# ============================================================
# STEP 2: DISPLAY INITIAL RECORDS
# ============================================================

print("=" * 80)
print("FIRST 5 RECORDS")
print("=" * 80)

print(df.head())

print("=" * 80)

FIRST 5 RECORDS
      BeneID   ClaimID ClaimStartDt  ClaimEndDt  Provider  \
0  BENE11001  CLM46614   2009-04-12  2009-04-18  PRV55912   
1  BENE11001  CLM66048   2009-08-31  2009-09-02  PRV55907   
2  BENE11001  CLM68358   2009-09-17  2009-09-20  PRV56046   
3  BENE11011  CLM38412   2009-02-14  2009-02-22  PRV52405   
4  BENE11014  CLM63689   2009-08-13  2009-08-30  PRV56614   

   InscClaimAmtReimbursed AttendingPhysician OperatingPhysician  \
0                   26000          PHY390922                NaN   
1                    5000          PHY318495          PHY318495   
2                    5000          PHY372395                NaN   
3                    5000          PHY369659          PHY392961   
4                   10000          PHY379376          PHY398258   

  OtherPhysician AdmissionDt  ... ClmDiagnosisCode_7  ClmDiagnosisCode_8  \
0            NaN  2009-04-12  ...               2724               19889   
1            NaN  2009-08-31  ...                NaN          

## 3. Column Inventory

A complete list of columns is generated to understand the available
information in the inpatient dataset.

The columns can later be classified into:

1. Identifiers
2. Date fields
3. Financial/numerical fields
4. Physician fields
5. Diagnosis fields
6. Procedure fields

In [4]:
# ============================================================
# STEP 3: COLUMN INVENTORY
# ============================================================

print("=" * 80)
print("COLUMN INVENTORY")
print("=" * 80)

for i, column in enumerate(df.columns, start=1):
    print(f"{i:02d}. {column}")

print("=" * 80)

COLUMN INVENTORY
01. BeneID
02. ClaimID
03. ClaimStartDt
04. ClaimEndDt
05. Provider
06. InscClaimAmtReimbursed
07. AttendingPhysician
08. OperatingPhysician
09. OtherPhysician
10. AdmissionDt
11. ClmAdmitDiagnosisCode
12. DeductibleAmtPaid
13. DischargeDt
14. DiagnosisGroupCode
15. ClmDiagnosisCode_1
16. ClmDiagnosisCode_2
17. ClmDiagnosisCode_3
18. ClmDiagnosisCode_4
19. ClmDiagnosisCode_5
20. ClmDiagnosisCode_6
21. ClmDiagnosisCode_7
22. ClmDiagnosisCode_8
23. ClmDiagnosisCode_9
24. ClmDiagnosisCode_10
25. ClmProcedureCode_1
26. ClmProcedureCode_2
27. ClmProcedureCode_3
28. ClmProcedureCode_4
29. ClmProcedureCode_5
30. ClmProcedureCode_6


## 4. Column Classification

The columns are categorized according to their semantic role.

### Identifier columns

These identify entities or records:

- BeneID
- ClaimID
- Provider

### Date columns

These represent claim and hospitalization timelines:

- ClaimStartDt
- ClaimEndDt
- AdmissionDt
- DischargeDt

### Financial columns

- InscClaimAmtReimbursed
- DeductibleAmtPaid

### Physician columns

- AttendingPhysician
- OperatingPhysician
- OtherPhysician

### Diagnosis columns

- ClmAdmitDiagnosisCode
- DiagnosisGroupCode
- ClmDiagnosisCode_1 through ClmDiagnosisCode_10

### Procedure columns

- ClmProcedureCode_1 through ClmProcedureCode_6

In [5]:
# ============================================================
# STEP 4: CLASSIFY COLUMNS
# ============================================================

identifier_columns = [
    "BeneID",
    "ClaimID",
    "Provider"
]

date_columns = [
    "ClaimStartDt",
    "ClaimEndDt",
    "AdmissionDt",
    "DischargeDt"
]

financial_columns = [
    "InscClaimAmtReimbursed",
    "DeductibleAmtPaid"
]

physician_columns = [
    "AttendingPhysician",
    "OperatingPhysician",
    "OtherPhysician"
]

diagnosis_columns = [
    "ClmAdmitDiagnosisCode",
    "DiagnosisGroupCode"
] + [f"ClmDiagnosisCode_{i}" for i in range(1, 11)]

procedure_columns = [
    f"ClmProcedureCode_{i}" for i in range(1, 7)
]

print("=" * 80)
print("COLUMN CLASSIFICATION")
print("=" * 80)

print(f"Identifier columns : {len(identifier_columns)}")
print(f"Date columns       : {len(date_columns)}")
print(f"Financial columns  : {len(financial_columns)}")
print(f"Physician columns  : {len(physician_columns)}")
print(f"Diagnosis columns  : {len(diagnosis_columns)}")
print(f"Procedure columns  : {len(procedure_columns)}")

print("=" * 80)

COLUMN CLASSIFICATION
Identifier columns : 3
Date columns       : 4
Financial columns  : 2
Physician columns  : 3
Diagnosis columns  : 12
Procedure columns  : 6


## 5. Inspect Data Types

The data types are examined before transformation.

This is important because:

- Dates should be datetime objects
- Financial amounts should be numerical
- IDs should remain categorical/string values
- Diagnosis and procedure codes should be treated as codes,
  not continuous numerical measurements

In [6]:
# ============================================================
# STEP 5: CHECK DATA TYPES
# ============================================================

print("=" * 80)
print("CURRENT DATA TYPES")
print("=" * 80)

print(df.dtypes)

print("=" * 80)

CURRENT DATA TYPES
BeneID                     object
ClaimID                    object
ClaimStartDt               object
ClaimEndDt                 object
Provider                   object
InscClaimAmtReimbursed      int64
AttendingPhysician         object
OperatingPhysician         object
OtherPhysician             object
AdmissionDt                object
ClmAdmitDiagnosisCode      object
DeductibleAmtPaid         float64
DischargeDt                object
DiagnosisGroupCode         object
ClmDiagnosisCode_1         object
ClmDiagnosisCode_2         object
ClmDiagnosisCode_3         object
ClmDiagnosisCode_4         object
ClmDiagnosisCode_5         object
ClmDiagnosisCode_6         object
ClmDiagnosisCode_7         object
ClmDiagnosisCode_8         object
ClmDiagnosisCode_9         object
ClmDiagnosisCode_10        object
ClmProcedureCode_1        float64
ClmProcedureCode_2        float64
ClmProcedureCode_3        float64
ClmProcedureCode_4        float64
ClmProcedureCode_5        flo

## 6. Duplicate Record Detection

Duplicate records can cause the same claim to be counted multiple times.

Therefore, complete-row duplicates are checked before further processing.

ClaimID uniqueness is also checked because ClaimID represents an individual
claim record.

In [7]:
# ============================================================
# STEP 6: DUPLICATE CHECK
# ============================================================

duplicate_rows = df.duplicated().sum()
duplicate_claim_ids = df["ClaimID"].duplicated().sum()

print("=" * 80)
print("DUPLICATE ANALYSIS")
print("=" * 80)

print(f"Duplicate complete rows : {duplicate_rows:,}")
print(f"Duplicate ClaimIDs      : {duplicate_claim_ids:,}")

print("=" * 80)

DUPLICATE ANALYSIS
Duplicate complete rows : 0
Duplicate ClaimIDs      : 0


## 7. Missing Value Analysis

Missing values are analyzed column-by-column.

The purpose is not to automatically replace every missing value.

Instead, we determine:

- How many values are missing
- What percentage is missing
- Whether the missingness is expected
- Whether the column should be retained
- Whether imputation is appropriate

In [8]:
# ============================================================
# STEP 7: MISSING VALUE ANALYSIS
# ============================================================

missing_count = df.isna().sum()
missing_percentage = (missing_count / len(df)) * 100

missing_summary = pd.DataFrame({
    "Missing_Count": missing_count,
    "Missing_Percentage": missing_percentage
})

missing_summary = missing_summary.sort_values(
    by="Missing_Percentage",
    ascending=False
)

print("=" * 80)
print("MISSING VALUE ANALYSIS")
print("=" * 80)

print(missing_summary.to_string())

print("=" * 80)

MISSING VALUE ANALYSIS
                        Missing_Count  Missing_Percentage
ClmProcedureCode_6              40474          100.000000
ClmProcedureCode_5              40465           99.977764
ClmProcedureCode_4              40358           99.713396
ClmProcedureCode_3              39509           97.615753
ClmDiagnosisCode_10             36547           90.297475
OtherPhysician                  35784           88.412314
ClmProcedureCode_2              35020           86.524683
ClmProcedureCode_1              17326           42.807728
OperatingPhysician              16644           41.122696
ClmDiagnosisCode_9              13497           33.347334
ClmDiagnosisCode_8               9942           24.563918
ClmDiagnosisCode_7               7258           17.932500
ClmDiagnosisCode_6               4838           11.953353
ClmDiagnosisCode_5               2894            7.150269
ClmDiagnosisCode_4               1534            3.790087
DeductibleAmtPaid                 899            

In [9]:
# ============================================================
# STEP 8: MISSINGNESS BY COLUMN CATEGORY
# ============================================================

def print_missingness(columns, category_name):
    print("\n" + "-" * 80)
    print(category_name)
    print("-" * 80)

    for col in columns:
        count = df[col].isna().sum()
        percentage = (count / len(df)) * 100

        print(
            f"{col:<30} "
            f"Missing: {count:>6,} "
            f"({percentage:>6.2f}%)"
        )

print("=" * 80)
print("MISSINGNESS BY CATEGORY")
print("=" * 80)

print_missingness(physician_columns, "PHYSICIAN COLUMNS")
print_missingness(diagnosis_columns, "DIAGNOSIS COLUMNS")
print_missingness(procedure_columns, "PROCEDURE COLUMNS")
print_missingness(financial_columns, "FINANCIAL COLUMNS")

print("=" * 80)

MISSINGNESS BY CATEGORY

--------------------------------------------------------------------------------
PHYSICIAN COLUMNS
--------------------------------------------------------------------------------
AttendingPhysician             Missing:    112 (  0.28%)
OperatingPhysician             Missing: 16,644 ( 41.12%)
OtherPhysician                 Missing: 35,784 ( 88.41%)

--------------------------------------------------------------------------------
DIAGNOSIS COLUMNS
--------------------------------------------------------------------------------
ClmAdmitDiagnosisCode          Missing:      0 (  0.00%)
DiagnosisGroupCode             Missing:      0 (  0.00%)
ClmDiagnosisCode_1             Missing:      0 (  0.00%)
ClmDiagnosisCode_2             Missing:    226 (  0.56%)
ClmDiagnosisCode_3             Missing:    676 (  1.67%)
ClmDiagnosisCode_4             Missing:  1,534 (  3.79%)
ClmDiagnosisCode_5             Missing:  2,894 (  7.15%)
ClmDiagnosisCode_6             Missing:  4,8

## 9. Identifier Validation

The inpatient dataset contains three important identifiers:

- BeneID → identifies the beneficiary
- ClaimID → identifies the claim
- Provider → identifies the healthcare provider

These identifiers should not be treated as continuous numerical variables.

They are retained as categorical/string identifiers for integration and
later provider-level processing.

In [10]:
# ============================================================
# STEP 9: IDENTIFIER VALIDATION
# ============================================================

print("=" * 80)
print("IDENTIFIER ANALYSIS")
print("=" * 80)

for col in identifier_columns:

    unique_count = df[col].nunique(dropna=True)
    missing_count = df[col].isna().sum()

    print(f"\n{col}")
    print(f"  Unique values : {unique_count:,}")
    print(f"  Missing       : {missing_count:,}")

print("=" * 80)

IDENTIFIER ANALYSIS

BeneID
  Unique values : 31,289
  Missing       : 0

ClaimID
  Unique values : 40,474
  Missing       : 0

Provider
  Unique values : 2,092
  Missing       : 0


## 10. Date Conversion

The following columns represent dates:

- ClaimStartDt
- ClaimEndDt
- AdmissionDt
- DischargeDt

They are currently stored as object/string values.

They will be converted into pandas datetime format.

This allows reliable chronological validation and prevents dates from
being incorrectly treated as ordinary strings.

In [11]:
# ============================================================
# STEP 10: CONVERT DATE COLUMNS
# ============================================================

print("=" * 80)
print("DATE CONVERSION")
print("=" * 80)

for col in date_columns:

    # Convert string/object date values to datetime
    df[col] = pd.to_datetime(
        df[col],
        errors="coerce"
    )

    print(f"{col:<20} → {df[col].dtype}")

print("=" * 80)

DATE CONVERSION
ClaimStartDt         → datetime64[ns]
ClaimEndDt           → datetime64[ns]
AdmissionDt          → datetime64[ns]
DischargeDt          → datetime64[ns]


## 11. Date Consistency Validation

After converting dates into datetime format, chronological relationships
between the dates are checked.

The following relationships are validated:

1. ClaimEndDt should not occur before ClaimStartDt.
2. DischargeDt should not occur before AdmissionDt.
3. DischargeDt should not occur before ClaimStartDt.
4. AdmissionDt may precede ClaimStartDt depending on the claim recording
   process, so this condition is investigated rather than automatically
   treated as an error.

In [12]:
# ============================================================
# STEP 11: DATE CONSISTENCY CHECK
# ============================================================

claim_end_before_start = (
    df["ClaimEndDt"] < df["ClaimStartDt"]
).sum()

discharge_before_admission = (
    df["DischargeDt"] < df["AdmissionDt"]
).sum()

discharge_before_claim_start = (
    df["DischargeDt"] < df["ClaimStartDt"]
).sum()

admission_before_claim_start = (
    df["AdmissionDt"] < df["ClaimStartDt"]
).sum()

admission_after_claim_end = (
    df["AdmissionDt"] > df["ClaimEndDt"]
).sum()

print("=" * 80)
print("DATE CONSISTENCY CHECK")
print("=" * 80)

print(
    f"ClaimEndDt < ClaimStartDt       : "
    f"{claim_end_before_start:,}"
)

print(
    f"DischargeDt < AdmissionDt       : "
    f"{discharge_before_admission:,}"
)

print(
    f"DischargeDt < ClaimStartDt      : "
    f"{discharge_before_claim_start:,}"
)

print(
    f"AdmissionDt < ClaimStartDt      : "
    f"{admission_before_claim_start:,}"
)

print(
    f"AdmissionDt > ClaimEndDt         : "
    f"{admission_after_claim_end:,}"
)

print("=" * 80)

DATE CONSISTENCY CHECK
ClaimEndDt < ClaimStartDt       : 0
DischargeDt < AdmissionDt       : 0
DischargeDt < ClaimStartDt      : 0
AdmissionDt < ClaimStartDt      : 32
AdmissionDt > ClaimEndDt         : 0


## 12. Numerical Value Validation

The financial columns are examined for:

- Minimum value
- Maximum value
- Mean
- Median
- Negative values
- Zero values
- Missing values

Negative reimbursement or deductible values would require investigation.

In [13]:
# ============================================================
# STEP 12: NUMERICAL VALIDATION
# ============================================================

print("=" * 80)
print("FINANCIAL DATA VALIDATION")
print("=" * 80)

for col in financial_columns:

    print("\n" + "-" * 80)
    print(f"COLUMN: {col}")
    print("-" * 80)

    print(f"Data type       : {df[col].dtype}")
    print(f"Missing values  : {df[col].isna().sum():,}")
    print(f"Minimum         : {df[col].min()}")
    print(f"Maximum         : {df[col].max()}")
    print(f"Mean            : {df[col].mean():,.2f}")
    print(f"Median          : {df[col].median():,.2f}")
    print(f"Negative values : {(df[col] < 0).sum():,}")
    print(f"Zero values     : {(df[col] == 0).sum():,}")

print("=" * 80)

FINANCIAL DATA VALIDATION

--------------------------------------------------------------------------------
COLUMN: InscClaimAmtReimbursed
--------------------------------------------------------------------------------
Data type       : int64
Missing values  : 0
Minimum         : 0
Maximum         : 125000
Mean            : 10,087.88
Median          : 7,000.00
Negative values : 0
Zero values     : 1,085

--------------------------------------------------------------------------------
COLUMN: DeductibleAmtPaid
--------------------------------------------------------------------------------
Data type       : float64
Missing values  : 899
Minimum         : 1068.0
Maximum         : 1068.0
Mean            : 1,068.00
Median          : 1,068.00
Negative values : 0
Zero values     : 0


## 13. Standardize Diagnosis and Procedure Code Types

Diagnosis and procedure codes represent medical classification codes.

They should not be treated as continuous numerical measurements.

Procedure columns are currently loaded as float because they contain missing
values.

They will therefore be converted to string-based code fields.

This prevents values such as 7092.0 from being interpreted as numerical
measurements and prepares the data for later integration and feature
engineering.

In [14]:
# ============================================================
# STEP 13: STANDARDIZE CODE COLUMNS
# ============================================================

print("=" * 80)
print("STANDARDIZING DIAGNOSIS AND PROCEDURE CODE TYPES")
print("=" * 80)

# ------------------------------------------------------------
# Diagnosis codes
# ------------------------------------------------------------

for col in diagnosis_columns:

    df[col] = df[col].astype("string").str.strip()

# ------------------------------------------------------------
# Procedure codes
# ------------------------------------------------------------

for col in procedure_columns:

    # Convert numerical procedure codes such as 7092.0
    # into string code representation such as "7092".
    df[col] = (
        df[col]
        .astype("Int64")
        .astype("string")
    )

print("\nCode columns converted to string representation.")

print("\nUpdated procedure column data types:")
print(df[procedure_columns].dtypes)

print("=" * 80)

STANDARDIZING DIAGNOSIS AND PROCEDURE CODE TYPES

Code columns converted to string representation.

Updated procedure column data types:
ClmProcedureCode_1    string[python]
ClmProcedureCode_2    string[python]
ClmProcedureCode_3    string[python]
ClmProcedureCode_4    string[python]
ClmProcedureCode_5    string[python]
ClmProcedureCode_6    string[python]
dtype: object


## 14. Standardize Identifier Columns

Identifier fields are converted to string representation.

This ensures that identifiers remain categorical labels rather than
numerical measurements.

The identifiers will later be used for dataset integration.

In [15]:
# ============================================================
# STEP 14: STANDARDIZE IDENTIFIERS
# ============================================================

print("=" * 80)
print("STANDARDIZING IDENTIFIER COLUMNS")
print("=" * 80)

for col in identifier_columns:

    df[col] = df[col].astype("string").str.strip()

print(df[identifier_columns].dtypes)

print("=" * 80)

STANDARDIZING IDENTIFIER COLUMNS
BeneID      string[python]
ClaimID     string[python]
Provider    string[python]
dtype: object


## 15. Standardize Physician Identifier Columns

The inpatient dataset contains three physician-related identifier fields:

- AttendingPhysician
- OperatingPhysician
- OtherPhysician

These fields identify physicians associated with an inpatient claim.

Since physician IDs are identifiers rather than numerical measurements, they are
converted to string representation.

Missing physician identifiers are preserved because a missing value may indicate
that a particular physician role was not recorded or was not applicable to the
claim.

No artificial values are introduced for missing physician identifiers.

In [16]:
# ============================================================
# STEP 15: STANDARDIZE PHYSICIAN IDENTIFIER COLUMNS
# ============================================================

physician_columns = [
    "AttendingPhysician",
    "OperatingPhysician",
    "OtherPhysician"
]

print("=" * 80)
print("STANDARDIZING PHYSICIAN IDENTIFIER COLUMNS")
print("=" * 80)

for col in physician_columns:

    # Convert physician identifiers to string representation.
    # Missing values remain missing.
    df[col] = df[col].astype("string").str.strip()

    missing_count = df[col].isna().sum()
    missing_percentage = (missing_count / len(df)) * 100

    print(f"\nColumn: {col}")
    print(f"Data type        : {df[col].dtype}")
    print(f"Missing values   : {missing_count:,}")
    print(f"Missing %        : {missing_percentage:.2f}%")

print("=" * 80)

STANDARDIZING PHYSICIAN IDENTIFIER COLUMNS

Column: AttendingPhysician
Data type        : string
Missing values   : 112
Missing %        : 0.28%

Column: OperatingPhysician
Data type        : string
Missing values   : 16,644
Missing %        : 41.12%

Column: OtherPhysician
Data type        : string
Missing values   : 35,784
Missing %        : 88.41%


### Observation

All physician identifier columns were standardized as string fields.

Missing physician identifiers were retained because their absence does not
necessarily indicate an invalid claim. A physician role may not have been
recorded or may not have been applicable to a particular claim.

No rows were removed and no artificial physician identifiers were created.

## 16. Standardize Diagnosis Code Columns

Diagnosis-related fields contain medical classification codes associated with
each inpatient claim.

The diagnosis columns include:

- ClmAdmitDiagnosisCode
- DiagnosisGroupCode
- ClmDiagnosisCode_1 through ClmDiagnosisCode_10

These values represent codes rather than continuous numerical measurements.
Therefore, they are stored as string values.

Missing diagnosis positions are preserved because a claim may contain fewer
diagnosis codes than the maximum number of available diagnosis fields.

No diagnosis values are artificially imputed during this preprocessing stage.

In [17]:
# ============================================================
# STEP 16: STANDARDIZE DIAGNOSIS CODE COLUMNS
# ============================================================

diagnosis_columns = [
    "ClmAdmitDiagnosisCode",
    "DiagnosisGroupCode"
] + [f"ClmDiagnosisCode_{i}" for i in range(1, 11)]

print("=" * 80)
print("STANDARDIZING DIAGNOSIS CODE COLUMNS")
print("=" * 80)

for col in diagnosis_columns:

    # Convert diagnosis codes to string representation.
    # Missing values remain missing.
    df[col] = df[col].astype("string").str.strip()

print("\nDiagnosis column data types:")
print(df[diagnosis_columns].dtypes)

print("=" * 80)

STANDARDIZING DIAGNOSIS CODE COLUMNS

Diagnosis column data types:
ClmAdmitDiagnosisCode    string[python]
DiagnosisGroupCode       string[python]
ClmDiagnosisCode_1       string[python]
ClmDiagnosisCode_2       string[python]
ClmDiagnosisCode_3       string[python]
ClmDiagnosisCode_4       string[python]
ClmDiagnosisCode_5       string[python]
ClmDiagnosisCode_6       string[python]
ClmDiagnosisCode_7       string[python]
ClmDiagnosisCode_8       string[python]
ClmDiagnosisCode_9       string[python]
ClmDiagnosisCode_10      string[python]
dtype: object


## 17. Standardize Procedure Code Columns

Procedure-related fields contain medical procedure codes associated with
inpatient claims.

The procedure codes were initially represented as floating-point values in
some columns because the presence of missing values caused automatic numeric
type inference.

Procedure codes should not be treated as continuous numerical measurements.
They are therefore converted into string-based code representation.

Missing procedure codes are preserved.

A completely empty procedure column will be evaluated separately during the
missing-value analysis.

In [18]:
# ============================================================
# STEP 17: STANDARDIZE PROCEDURE CODE COLUMNS
# ============================================================

procedure_columns = [
    f"ClmProcedureCode_{i}" for i in range(1, 7)
]

print("=" * 80)
print("STANDARDIZING PROCEDURE CODE COLUMNS")
print("=" * 80)

for col in procedure_columns:

    # Convert to nullable integer first so values such as 7092.0
    # become 7092 while missing values remain <NA>.
    df[col] = df[col].astype("Int64")

    # Convert medical procedure codes to string representation.
    df[col] = df[col].astype("string")

print("\nProcedure column data types:")
print(df[procedure_columns].dtypes)

print("=" * 80)

STANDARDIZING PROCEDURE CODE COLUMNS

Procedure column data types:
ClmProcedureCode_1    string[python]
ClmProcedureCode_2    string[python]
ClmProcedureCode_3    string[python]
ClmProcedureCode_4    string[python]
ClmProcedureCode_5    string[python]
ClmProcedureCode_6    string[python]
dtype: object


## 18. Identify Completely Empty Columns

A completely empty column is a column in which every record contains a
missing value.

Such a column provides no information in the current dataset and can be
considered for removal.

This check is different from ordinary missing-value analysis.

A column with 90% missing values is not completely empty and should not be
automatically removed at this stage.

In [19]:
# ============================================================
# STEP 18: IDENTIFY COMPLETELY EMPTY COLUMNS
# ============================================================

empty_columns = [
    col for col in df.columns
    if df[col].isna().all()
]

print("=" * 80)
print("COMPLETELY EMPTY COLUMNS")
print("=" * 80)

if len(empty_columns) == 0:

    print("No completely empty columns found.")

else:

    print(f"Number of completely empty columns: {len(empty_columns)}")
    print()

    for col in empty_columns:
        print(f"- {col}")

print("=" * 80)

COMPLETELY EMPTY COLUMNS
Number of completely empty columns: 1

- ClmProcedureCode_6


### Observation

ClmProcedureCode_6 contains no observed values across the entire inpatient
dataset.

Since the column contains no information in the current dataset, it is
removed from the cleaned inpatient dataset.

Other columns with partial missingness are retained for further analysis.

In [20]:
# Remove only the completely empty column
df.drop(columns=empty_columns, inplace=True)

## 19. Detailed Missing Value Analysis

Missing values are analyzed after standardizing the identifier, physician,
diagnosis and procedure fields.

The purpose is to understand the extent and distribution of missingness
before deciding whether any treatment is appropriate.

Missing values will not be automatically replaced with zero, mean, median,
or mode because the meaning of missingness differs between identifiers,
medical codes, physician fields and financial fields.

In [21]:
# ============================================================
# STEP 19: DETAILED MISSING VALUE ANALYSIS
# ============================================================

missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (
        df.isna().sum() / len(df) * 100
    )
})

missing_summary = missing_summary.sort_values(
    by="Missing_Percentage",
    ascending=False
)

print("=" * 80)
print("DETAILED MISSING VALUE ANALYSIS")
print("=" * 80)

print(missing_summary.to_string())

print("=" * 80)

DETAILED MISSING VALUE ANALYSIS
                        Missing_Count  Missing_Percentage
ClmProcedureCode_5              40465           99.977764
ClmProcedureCode_4              40358           99.713396
ClmProcedureCode_3              39509           97.615753
ClmDiagnosisCode_10             36547           90.297475
OtherPhysician                  35784           88.412314
ClmProcedureCode_2              35020           86.524683
ClmProcedureCode_1              17326           42.807728
OperatingPhysician              16644           41.122696
ClmDiagnosisCode_9              13497           33.347334
ClmDiagnosisCode_8               9942           24.563918
ClmDiagnosisCode_7               7258           17.932500
ClmDiagnosisCode_6               4838           11.953353
ClmDiagnosisCode_5               2894            7.150269
ClmDiagnosisCode_4               1534            3.790087
DeductibleAmtPaid                 899            2.221179
ClmDiagnosisCode_3                676   

In [22]:
# ============================================================
# STEP 20: INSPECT FINANCIAL MISSING VALUES
# ============================================================

print("=" * 80)
print("DEDUCTIBLE AMOUNT ANALYSIS")
print("=" * 80)

print("Missing values:")
print(df["DeductibleAmtPaid"].isna().sum())

print("\nDescriptive statistics:")
print(df["DeductibleAmtPaid"].describe())

print("\nUnique observed values:")
print(df["DeductibleAmtPaid"].dropna().unique())

print("=" * 80)

DEDUCTIBLE AMOUNT ANALYSIS
Missing values:
899

Descriptive statistics:
count    39575.0
mean      1068.0
std          0.0
min       1068.0
25%       1068.0
50%       1068.0
75%       1068.0
max       1068.0
Name: DeductibleAmtPaid, dtype: float64

Unique observed values:
[1068.]


## Missing Value Interpretation

The missing-value analysis shows that missingness is concentrated primarily
in optional physician, diagnosis and procedure fields.

The later diagnosis and procedure slots have substantially higher missingness
than the earlier slots. This is expected from the structure of claim-level
medical coding, where individual claims may contain different numbers of
diagnoses and procedures.

Therefore, high missingness alone is not used as a reason to remove these
columns.

Physician identifier fields are also retained because a missing physician
value may indicate that the physician role was not recorded or was not
applicable to the claim.

The only column identified as completely empty was
`ClmProcedureCode_6`, which was removed after verification.

No blanket mean, median, mode, or zero imputation is performed at this stage.
Further treatment of missing values will be determined based on the role of
each variable in the final unified dataset and modeling pipeline.

## 20. Remove Constant-Valued Columns

`DeductibleAmtPaid` was analyzed to determine whether the variable contains
meaningful variation.

The column contains 899 missing values. Among the 39,575 non-missing records,
every observed value is exactly 1068.

Therefore:

- Mean = 1068
- Median = 1068
- Minimum = 1068
- Maximum = 1068
- Standard deviation = 0
- Number of unique observed values = 1

Since the column has no variation among observed records, it cannot provide
useful discriminative information in its current form.

The column is therefore removed from the cleaned inpatient dataset.

The missing values are not imputed because doing so would result in a
completely constant column.

In [23]:
# ============================================================
# STEP 20: REMOVE CONSTANT-VALUED COLUMN
# ============================================================

print("=" * 80)
print("CONSTANT-VALUE COLUMN HANDLING")
print("=" * 80)

column = "DeductibleAmtPaid"

unique_values = df[column].dropna().nunique()
std_value = df[column].std()

print(f"Column                     : {column}")
print(f"Observed unique values     : {unique_values}")
print(f"Standard deviation         : {std_value}")
print(f"Missing values             : {df[column].isna().sum():,}")

# Remove the column because all observed values are identical.
if unique_values == 1 and std_value == 0:

    df.drop(columns=[column], inplace=True)

    print("\nDecision:")
    print(f"✓ {column} removed because it has zero variance.")

else:

    print("\nDecision:")
    print(f"✓ {column} retained because it contains variation.")

print("=" * 80)

CONSTANT-VALUE COLUMN HANDLING
Column                     : DeductibleAmtPaid
Observed unique values     : 1
Standard deviation         : 0.0
Missing values             : 899

Decision:
✓ DeductibleAmtPaid removed because it has zero variance.


## 21. Validate Inpatient Reimbursement Amount

`InscClaimAmtReimbursed` represents the reimbursement amount associated
with each inpatient claim.

The column is validated for:

- Missing values
- Negative values
- Zero values
- Minimum and maximum values
- Mean and median
- Standard deviation

Negative reimbursement values would require investigation because they may
represent invalid or unusual records.

In [24]:
# ============================================================
# STEP 21: VALIDATE REIMBURSEMENT AMOUNT
# ============================================================

column = "InscClaimAmtReimbursed"

print("=" * 80)
print("INPATIENT REIMBURSEMENT VALIDATION")
print("=" * 80)

print(f"Column            : {column}")
print(f"Data type         : {df[column].dtype}")
print(f"Missing values    : {df[column].isna().sum():,}")
print(f"Negative values   : {(df[column] < 0).sum():,}")
print(f"Zero values       : {(df[column] == 0).sum():,}")
print(f"Minimum           : {df[column].min():,.2f}")
print(f"Maximum           : {df[column].max():,.2f}")
print(f"Mean              : {df[column].mean():,.2f}")
print(f"Median            : {df[column].median():,.2f}")
print(f"Standard deviation: {df[column].std():,.2f}")

print("=" * 80)

INPATIENT REIMBURSEMENT VALIDATION
Column            : InscClaimAmtReimbursed
Data type         : int64
Missing values    : 0
Negative values   : 0
Zero values       : 1,085
Minimum           : 0.00
Maximum           : 125,000.00
Mean              : 10,087.88
Median            : 7,000.00
Standard deviation: 10,303.10


## 22. Date Consistency Validation

The inpatient claims dataset contains four date fields:

- ClaimStartDt
- ClaimEndDt
- AdmissionDt
- DischargeDt

After converting the fields to datetime format, chronological consistency
is validated.

The following conditions are checked:

1. ClaimEndDt should not occur before ClaimStartDt.
2. DischargeDt should not occur before AdmissionDt.
3. DischargeDt should not occur before ClaimStartDt.
4. AdmissionDt should not occur after ClaimEndDt.
5. Records where AdmissionDt occurs before ClaimStartDt are identified for
   review because the difference may reflect claim-recording timing rather
   than an invalid hospitalization.

Invalid-looking records are not automatically deleted without sufficient
evidence that the underlying data is erroneous.

In [25]:
# ============================================================
# STEP 22: FINAL DATE CONSISTENCY VALIDATION
# ============================================================

date_columns = [
    "ClaimStartDt",
    "ClaimEndDt",
    "AdmissionDt",
    "DischargeDt"
]

print("=" * 80)
print("FINAL DATE CONSISTENCY VALIDATION")
print("=" * 80)

# ------------------------------------------------------------
# Check whether all date columns are actually datetime
# ------------------------------------------------------------

print("\nDATE DATA TYPES")
print("-" * 80)

for col in date_columns:
    print(f"{col:<20} : {df[col].dtype}")

# ------------------------------------------------------------
# Chronological consistency checks
# ------------------------------------------------------------

claim_end_before_start = (
    df["ClaimEndDt"] < df["ClaimStartDt"]
).sum()

discharge_before_admission = (
    df["DischargeDt"] < df["AdmissionDt"]
).sum()

discharge_before_claim_start = (
    df["DischargeDt"] < df["ClaimStartDt"]
).sum()

admission_after_claim_end = (
    df["AdmissionDt"] > df["ClaimEndDt"]
).sum()

admission_before_claim_start = (
    df["AdmissionDt"] < df["ClaimStartDt"]
).sum()

# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("\nCHRONOLOGICAL VALIDATION")
print("-" * 80)

print(
    f"ClaimEndDt < ClaimStartDt       : "
    f"{claim_end_before_start:,}"
)

print(
    f"DischargeDt < AdmissionDt       : "
    f"{discharge_before_admission:,}"
)

print(
    f"DischargeDt < ClaimStartDt      : "
    f"{discharge_before_claim_start:,}"
)

print(
    f"AdmissionDt > ClaimEndDt        : "
    f"{admission_after_claim_end:,}"
)

print(
    f"AdmissionDt < ClaimStartDt      : "
    f"{admission_before_claim_start:,}"
)

print("=" * 80)

FINAL DATE CONSISTENCY VALIDATION

DATE DATA TYPES
--------------------------------------------------------------------------------
ClaimStartDt         : datetime64[ns]
ClaimEndDt           : datetime64[ns]
AdmissionDt          : datetime64[ns]
DischargeDt          : datetime64[ns]

CHRONOLOGICAL VALIDATION
--------------------------------------------------------------------------------
ClaimEndDt < ClaimStartDt       : 0
DischargeDt < AdmissionDt       : 0
DischargeDt < ClaimStartDt      : 0
AdmissionDt > ClaimEndDt        : 0
AdmissionDt < ClaimStartDt      : 32


### Observation

The date validation found no records where:

- ClaimEndDt occurred before ClaimStartDt
- DischargeDt occurred before AdmissionDt
- DischargeDt occurred before ClaimStartDt
- AdmissionDt occurred after ClaimEndDt

There were 32 records where AdmissionDt occurred before ClaimStartDt.

These records are retained because this difference does not by itself prove
that the claim is invalid. Admission date and claim start date may represent
different events in the claims data.

Therefore, no records were removed based solely on this date relationship.

## 23. Final Data Type Validation

After the preprocessing transformations, the datatype of every column is
validated.

The purpose of this step is to ensure that identifiers, medical codes, dates,
and numerical variables are represented according to their semantic meaning.

Expected representations:

- Identifier fields → string
- Physician identifiers → string
- Diagnosis codes → string
- Procedure codes → string
- Date fields → datetime
- Financial fields → numerical

This validation is particularly important because the cleaned inpatient
dataset will later be integrated with the cleaned outpatient and beneficiary
datasets.

In [26]:
# ============================================================
# STEP 23: FINAL DATA TYPE VALIDATION
# ============================================================

print("=" * 80)
print("FINAL DATA TYPE VALIDATION")
print("=" * 80)

print("\nIDENTIFIER COLUMNS")
print("-" * 80)

for col in ["BeneID", "ClaimID", "Provider"]:
    print(f"{col:<30} : {df[col].dtype}")

print("\nPHYSICIAN COLUMNS")
print("-" * 80)

for col in [
    "AttendingPhysician",
    "OperatingPhysician",
    "OtherPhysician"
]:
    print(f"{col:<30} : {df[col].dtype}")

print("\nDATE COLUMNS")
print("-" * 80)

for col in [
    "ClaimStartDt",
    "ClaimEndDt",
    "AdmissionDt",
    "DischargeDt"
]:
    print(f"{col:<30} : {df[col].dtype}")

print("\nFINANCIAL COLUMNS")
print("-" * 80)

for col in ["InscClaimAmtReimbursed"]:
    print(f"{col:<30} : {df[col].dtype}")

print("\nDIAGNOSIS COLUMNS")
print("-" * 80)

for col in diagnosis_columns:
    if col in df.columns:
        print(f"{col:<30} : {df[col].dtype}")

print("\nPROCEDURE COLUMNS")
print("-" * 80)

for col in procedure_columns:
    if col in df.columns:
        print(f"{col:<30} : {df[col].dtype}")

print("=" * 80)

FINAL DATA TYPE VALIDATION

IDENTIFIER COLUMNS
--------------------------------------------------------------------------------
BeneID                         : string
ClaimID                        : string
Provider                       : string

PHYSICIAN COLUMNS
--------------------------------------------------------------------------------
AttendingPhysician             : string
OperatingPhysician             : string
OtherPhysician                 : string

DATE COLUMNS
--------------------------------------------------------------------------------
ClaimStartDt                   : datetime64[ns]
ClaimEndDt                     : datetime64[ns]
AdmissionDt                    : datetime64[ns]
DischargeDt                    : datetime64[ns]

FINANCIAL COLUMNS
--------------------------------------------------------------------------------
InscClaimAmtReimbursed         : int64

DIAGNOSIS COLUMNS
--------------------------------------------------------------------------------
ClmAdm

### Observation

All major column groups have been converted to appropriate data types.

Identifiers and medical codes are represented as strings, dates are stored
as datetime values, and the reimbursement amount remains numerical.

The datatype structure is now suitable for subsequent integration with the
outpatient and beneficiary datasets.

## 24. Standardize Missing Diagnosis and Physician Values

Missing values in diagnosis and physician identifier fields are converted to
the explicit categorical value `UNKNOWN`.

This approach preserves all claim records while making the absence of a
recorded value explicit.

`UNKNOWN` indicates that the source dataset did not contain a recorded value.
It does not indicate that a diagnosis, procedure, or physician was confirmed
to be absent.

This representation is also used in the outpatient preprocessing pipeline,
ensuring consistency between the inpatient and outpatient datasets before
integration.

In [27]:
# ============================================================
# STEP 24: HANDLE MISSING DIAGNOSIS AND PHYSICIAN VALUES
# ============================================================

print("=" * 80)
print("HANDLING MISSING DIAGNOSIS AND PHYSICIAN VALUES")
print("=" * 80)

# ------------------------------------------------------------
# Diagnosis columns
# ------------------------------------------------------------

print("\nDIAGNOSIS COLUMNS")
print("-" * 80)

for col in diagnosis_columns:

    if col in df.columns:

        before = df[col].isna().sum()

        # Replace missing diagnosis values with explicit
        # categorical representation.
        df[col] = df[col].fillna("UNKNOWN")

        after = df[col].isna().sum()

        print(
            f"{col:<30} "
            f"Before: {before:>6,} | "
            f"After: {after:>6,}"
        )

# ------------------------------------------------------------
# Physician columns
# ------------------------------------------------------------

print("\nPHYSICIAN COLUMNS")
print("-" * 80)

for col in physician_columns:

    before = df[col].isna().sum()

    # Replace missing physician identifiers with UNKNOWN.
    df[col] = df[col].fillna("UNKNOWN")

    after = df[col].isna().sum()

    print(
        f"{col:<30} "
        f"Before: {before:>6,} | "
        f"After: {after:>6,}"
    )

print("=" * 80)

HANDLING MISSING DIAGNOSIS AND PHYSICIAN VALUES

DIAGNOSIS COLUMNS
--------------------------------------------------------------------------------
ClmAdmitDiagnosisCode          Before:      0 | After:      0
DiagnosisGroupCode             Before:      0 | After:      0
ClmDiagnosisCode_1             Before:      0 | After:      0
ClmDiagnosisCode_2             Before:    226 | After:      0
ClmDiagnosisCode_3             Before:    676 | After:      0
ClmDiagnosisCode_4             Before:  1,534 | After:      0
ClmDiagnosisCode_5             Before:  2,894 | After:      0
ClmDiagnosisCode_6             Before:  4,838 | After:      0
ClmDiagnosisCode_7             Before:  7,258 | After:      0
ClmDiagnosisCode_8             Before:  9,942 | After:      0
ClmDiagnosisCode_9             Before: 13,497 | After:      0
ClmDiagnosisCode_10            Before: 36,547 | After:      0

PHYSICIAN COLUMNS
--------------------------------------------------------------------------------
Attendin

## 25. Temporary Claim and Hospitalization Duration Validation

Claim duration and hospitalization duration were calculated temporarily to
validate the chronological consistency of the inpatient claim records.

These duration variables were created only for validation purposes and were
not added as permanent columns to the dataset.

### Claim Duration

Claim duration represents the number of days between the claim start date
and claim end date.

The calculation used:

`ClaimEndDt - ClaimStartDt`

### Hospitalization Duration

Hospitalization duration represents the number of days between the admission
date and discharge date.

The calculation used:

`DischargeDt - AdmissionDt`

### Validation Objective

The primary purpose of this step was to identify negative durations, which
would indicate an impossible chronological relationship between the
corresponding dates.

Both duration variables were checked for:

- Minimum duration
- Maximum duration
- Mean duration
- Number of negative durations

The temporary duration variables were not retained in the final inpatient
dataset because this step was performed as a data-quality validation rather
than feature engineering.

In [28]:
# ============================================================
# STEP 25: TEMPORARY CLAIM / HOSPITALIZATION DURATION CHECK
# ============================================================

print("=" * 80)
print("TEMPORARY INPATIENT DURATION VALIDATION")
print("=" * 80)

# Calculate temporarily for validation only.
claim_duration = (
    df["ClaimEndDt"] - df["ClaimStartDt"]
).dt.days

hospitalization_duration = (
    df["DischargeDt"] - df["AdmissionDt"]
).dt.days

print("\nCLAIM DURATION")
print("-" * 80)

print(f"Minimum : {claim_duration.min()} days")
print(f"Maximum : {claim_duration.max()} days")
print(f"Mean    : {claim_duration.mean():.2f} days")
print(f"Negative durations : {(claim_duration < 0).sum():,}")

print("\nHOSPITALIZATION DURATION")
print("-" * 80)

print(f"Minimum : {hospitalization_duration.min()} days")
print(f"Maximum : {hospitalization_duration.max()} days")
print(f"Mean    : {hospitalization_duration.mean():.2f} days")
print(f"Negative durations : {(hospitalization_duration < 0).sum():,}")

print("=" * 80)

TEMPORARY INPATIENT DURATION VALIDATION

CLAIM DURATION
--------------------------------------------------------------------------------
Minimum : 0 days
Maximum : 36 days
Mean    : 5.66 days
Negative durations : 0

HOSPITALIZATION DURATION
--------------------------------------------------------------------------------
Minimum : 0 days
Maximum : 35 days
Mean    : 5.67 days
Negative durations : 0


### Observation

The claim duration ranged from 0 to 36 days, with an average duration of
5.66 days.

The hospitalization duration ranged from 0 to 35 days, with an average
duration of 5.67 days.

No negative claim durations or negative hospitalization durations were
identified.

Therefore, no chronological anomalies were detected through these duration
checks. The temporary duration calculations were not retained as dataset
columns because they were used only for validation at this preprocessing
stage.

## 26. Current Missing Value Percentage

After the column-removal and datatype-standardization steps, the missing-value
percentage is recalculated to understand the current state of the inpatient
dataset before applying the agreed missing-value representation.

This check is performed before replacing missing diagnosis and physician
identifiers with `UNKNOWN`.

In [29]:
# ============================================================
# STEP 26: CURRENT MISSING VALUE PERCENTAGE
# ============================================================

print("=" * 80)
print("CURRENT MISSING VALUE ANALYSIS")
print("=" * 80)

# Calculate missing count and percentage
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (
        df.isna().sum() / len(df) * 100
    )
})

# Show only columns that currently contain missing values
missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values(
    by="Missing_Percentage",
    ascending=False
)

print(missing_summary.to_string())

print("=" * 80)

CURRENT MISSING VALUE ANALYSIS
                    Missing_Count  Missing_Percentage
ClmProcedureCode_5          40465           99.977764
ClmProcedureCode_4          40358           99.713396
ClmProcedureCode_3          39509           97.615753
ClmProcedureCode_2          35020           86.524683
ClmProcedureCode_1          17326           42.807728


## 27. Current Inpatient Dataset Dimensions Check

Before proceeding with the remaining preprocessing steps, the current
dimensions and column structure of the inpatient dataset were inspected.

This validation was performed to establish a clear checkpoint of the
dataset before further column-level preprocessing.

### Dataset Dimensions

- Rows: 40,474
- Columns: 28

### Current Column Structure

The dataset contains:

- Beneficiary and claim identifiers
- Provider identifier
- Claim start and end dates
- Reimbursement amount
- Physician identifiers
- Admission and discharge dates
- Admission diagnosis and diagnosis group
- Diagnosis codes 1–10
- Procedure codes 1–5

`DeductibleAmtPaid` had already been removed following its zero-variance
analysis.

`ClmProcedureCode_6` had already been removed because it contained no
observed values.

The remaining procedure columns were retained temporarily at this stage
pending the missing-value analysis and subsequent team-guided decision on
procedure-code retention.

In [30]:
# ============================================================
# CHECK CURRENT DATASET DIMENSIONS
# ============================================================

print("=" * 80)
print("CURRENT INPATIENT DATASET DIMENSIONS")
print("=" * 80)

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]:,}")

print("\nCurrent column names:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

print("=" * 80)

CURRENT INPATIENT DATASET DIMENSIONS
Rows    : 40,474
Columns : 28

Current column names:
01. BeneID
02. ClaimID
03. ClaimStartDt
04. ClaimEndDt
05. Provider
06. InscClaimAmtReimbursed
07. AttendingPhysician
08. OperatingPhysician
09. OtherPhysician
10. AdmissionDt
11. ClmAdmitDiagnosisCode
12. DischargeDt
13. DiagnosisGroupCode
14. ClmDiagnosisCode_1
15. ClmDiagnosisCode_2
16. ClmDiagnosisCode_3
17. ClmDiagnosisCode_4
18. ClmDiagnosisCode_5
19. ClmDiagnosisCode_6
20. ClmDiagnosisCode_7
21. ClmDiagnosisCode_8
22. ClmDiagnosisCode_9
23. ClmDiagnosisCode_10
24. ClmProcedureCode_1
25. ClmProcedureCode_2
26. ClmProcedureCode_3
27. ClmProcedureCode_4
28. ClmProcedureCode_5


### Observation

The inpatient dataset contained 40,474 records and 28 columns at this
checkpoint.

The dataset retained the claim-level structure, where each row represents
an inpatient claim.

At this stage, the procedure columns `ClmProcedureCode_1` through
`ClmProcedureCode_5` were still present. Their retention/removal was
evaluated separately using their missing-value percentages.

The `DeductibleAmtPaid` column was no longer present because it had been
identified as a zero-variance field during the earlier analysis.

`ClmProcedureCode_6` was also no longer present because it contained 100%
missing values.

This checkpoint was used as the baseline dataset structure before the
subsequent procedure-code cleaning decision.

## 28. Procedure Code Availability Analysis

The procedure-code fields were analyzed to determine the amount of usable
procedure information available in each column.

For each procedure-code column, the following were calculated:

- Number of observed (non-missing) procedure values
- Number of missing values
- Number of unique procedure codes

`ClmProcedureCode_6` was excluded from this analysis because it had already
been identified as completely empty and removed during the earlier
preprocessing stage.

This analysis was performed before making the final decision regarding
sparse procedure-code columns.

In [31]:
# ============================================================
# STEP 28: PROCEDURE CODE AVAILABILITY ANALYSIS
# ============================================================

print("=" * 80)
print("PROCEDURE CODE AVAILABILITY ANALYSIS")
print("=" * 80)

for col in procedure_columns:

    # Skip the removed completely empty column
    if col not in df.columns:
        continue

    observed_count = df[col].notna().sum()
    missing_count = df[col].isna().sum()
    unique_codes = df[col].dropna().nunique()

    print(f"\nColumn: {col}")
    print(f"Observed values : {observed_count:,}")
    print(f"Missing values  : {missing_count:,}")
    print(f"Unique codes    : {unique_codes:,}")

print("=" * 80)

PROCEDURE CODE AVAILABILITY ANALYSIS

Column: ClmProcedureCode_1
Observed values : 23,148
Missing values  : 17,326
Unique codes    : 1,117

Column: ClmProcedureCode_2
Observed values : 5,454
Missing values  : 35,020
Unique codes    : 297

Column: ClmProcedureCode_3
Observed values : 965
Missing values  : 39,509
Unique codes    : 154

Column: ClmProcedureCode_4
Observed values : 116
Missing values  : 40,358
Unique codes    : 48

Column: ClmProcedureCode_5
Observed values : 9
Missing values  : 40,465
Unique codes    : 6


### Observation

The procedure-code availability varies substantially across the procedure
slots.

`ClmProcedureCode_1` contains the highest amount of usable information, with
23,148 observed values and 1,117 unique procedure codes.

`ClmProcedureCode_2` contains 5,454 observed values and 297 unique codes,
indicating that it still contains a meaningful amount of information despite
high missingness.

`ClmProcedureCode_3` contains only 965 observed values and has very high
missingness.

`ClmProcedureCode_4` contains only 116 observed values.

`ClmProcedureCode_5` contains only 9 observed values and is therefore
extremely sparse.

These results were used together with the missing-value percentages and the
team lead's guideline that procedure columns with 90% or more missing values
should be removed.

Therefore:

- `ClmProcedureCode_1` → retained
- `ClmProcedureCode_2` → retained
- `ClmProcedureCode_3` → removed
- `ClmProcedureCode_4` → removed
- `ClmProcedureCode_5` → removed

`ClmProcedureCode_6` had already been removed because it contained no
observed values.

The retained procedure columns can subsequently be evaluated during feature
engineering to determine whether procedure-related information contributes
useful signal for provider-level fraud detection.

In [32]:
# ============================================================
# COMPLETE CURRENT INPATIENT DATA AUDIT
# ============================================================

print("=" * 100)
print("CURRENT INPATIENT DATASET - COMPLETE DATA AUDIT")
print("=" * 100)


# ------------------------------------------------------------
# 1. DATASET DIMENSIONS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("1. DATASET DIMENSIONS")
print("=" * 100)

print(f"Number of rows       : {df.shape[0]:,}")
print(f"Number of columns    : {df.shape[1]:,}")
print(f"Total cells         : {df.size:,}")


# ------------------------------------------------------------
# 2. COLUMN NAMES
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("2. CURRENT COLUMN NAMES")
print("=" * 100)

for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")


# ------------------------------------------------------------
# 3. COMPLETE COLUMN-WISE SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("3. COMPLETE COLUMN-WISE SUMMARY")
print("=" * 100)

column_summary = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Non_Null_Count": df.notna().sum().values,
    "Missing_Count": df.isna().sum().values,
    "Missing_Percentage": (
        df.isna().sum().values / len(df) * 100
    ),
    "Unique_Count": [
        df[col].nunique(dropna=True)
        for col in df.columns
    ]
})

print(column_summary.to_string(index=False))


# ------------------------------------------------------------
# 4. FIRST 5 RECORDS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("4. FIRST 5 RECORDS")
print("=" * 100)

print(df.head().to_string())


# ------------------------------------------------------------
# 5. LAST 5 RECORDS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("5. LAST 5 RECORDS")
print("=" * 100)

print(df.tail().to_string())


# ------------------------------------------------------------
# 6. DUPLICATE ANALYSIS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("6. DUPLICATE ANALYSIS")
print("=" * 100)

print(f"Duplicate complete rows : {df.duplicated().sum():,}")

if "ClaimID" in df.columns:
    print(
        f"Duplicate ClaimIDs     : "
        f"{df['ClaimID'].duplicated().sum():,}"
    )


# ------------------------------------------------------------
# 7. IDENTIFIER ANALYSIS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("7. IDENTIFIER ANALYSIS")
print("=" * 100)

identifier_columns = [
    "BeneID",
    "ClaimID",
    "Provider"
]

for col in identifier_columns:

    if col in df.columns:

        print(f"\n{col}")
        print(f"  Data type       : {df[col].dtype}")
        print(f"  Unique values   : {df[col].nunique(dropna=True):,}")
        print(f"  Missing values  : {df[col].isna().sum():,}")

        # Display a few example identifiers
        print(
            f"  Sample values   : "
            f"{df[col].dropna().unique()[:5]}"
        )


# ------------------------------------------------------------
# 8. DATE COLUMN ANALYSIS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("8. DATE COLUMN ANALYSIS")
print("=" * 100)

date_columns = [
    "ClaimStartDt",
    "ClaimEndDt",
    "AdmissionDt",
    "DischargeDt"
]

for col in date_columns:

    if col in df.columns:

        print(f"\n{col}")
        print(f"  Data type       : {df[col].dtype}")
        print(f"  Missing values  : {df[col].isna().sum():,}")
        print(f"  Minimum date    : {df[col].min()}")
        print(f"  Maximum date    : {df[col].max()}")


# ------------------------------------------------------------
# 9. FINANCIAL COLUMN ANALYSIS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("9. FINANCIAL COLUMN ANALYSIS")
print("=" * 100)

financial_columns = [
    "InscClaimAmtReimbursed"
]

for col in financial_columns:

    if col in df.columns:

        print(f"\n{col}")
        print(f"  Data type       : {df[col].dtype}")
        print(f"  Missing values  : {df[col].isna().sum():,}")
        print(f"  Unique values   : {df[col].nunique(dropna=True):,}")
        print(f"  Minimum         : {df[col].min():,.2f}")
        print(f"  Maximum         : {df[col].max():,.2f}")
        print(f"  Mean            : {df[col].mean():,.2f}")
        print(f"  Median          : {df[col].median():,.2f}")
        print(f"  Std deviation   : {df[col].std():,.2f}")
        print(f"  Zero values     : {(df[col] == 0).sum():,}")
        print(f"  Negative values : {(df[col] < 0).sum():,}")


# ------------------------------------------------------------
# 10. PHYSICIAN COLUMN ANALYSIS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("10. PHYSICIAN COLUMN ANALYSIS")
print("=" * 100)

physician_columns = [
    "AttendingPhysician",
    "OperatingPhysician",
    "OtherPhysician"
]

for col in physician_columns:

    if col in df.columns:

        print(f"\n{col}")
        print(f"  Data type       : {df[col].dtype}")
        print(f"  Missing values  : {df[col].isna().sum():,}")
        print(
            f"  UNKNOWN values  : "
            f"{(df[col] == 'UNKNOWN').sum():,}"
        )
        print(
            f"  Unique values   : "
            f"{df[col].nunique(dropna=True):,}"
        )


# ------------------------------------------------------------
# 11. DIAGNOSIS COLUMN ANALYSIS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("11. DIAGNOSIS COLUMN ANALYSIS")
print("=" * 100)

diagnosis_columns = [
    "ClmAdmitDiagnosisCode",
    "DiagnosisGroupCode"
] + [
    f"ClmDiagnosisCode_{i}"
    for i in range(1, 11)
]

for col in diagnosis_columns:

    if col in df.columns:

        print(f"\n{col}")
        print(f"  Data type       : {df[col].dtype}")
        print(f"  Missing values  : {df[col].isna().sum():,}")
        print(
            f"  UNKNOWN values  : "
            f"{(df[col] == 'UNKNOWN').sum():,}"
        )
        print(
            f"  Unique codes    : "
            f"{df[col].nunique(dropna=True):,}"
        )


# ------------------------------------------------------------
# 12. PROCEDURE COLUMN ANALYSIS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("12. PROCEDURE COLUMN ANALYSIS")
print("=" * 100)

procedure_columns = [
    f"ClmProcedureCode_{i}"
    for i in range(1, 7)
]

for col in procedure_columns:

    if col in df.columns:

        observed = df[col].notna().sum()
        missing = df[col].isna().sum()
        unique = df[col].nunique(dropna=True)

        print(f"\n{col}")
        print(f"  Data type       : {df[col].dtype}")
        print(f"  Observed values : {observed:,}")
        print(f"  Missing values  : {missing:,}")
        print(
            f"  Missing %       : "
            f"{missing / len(df) * 100:.2f}%"
        )
        print(f"  Unique codes    : {unique:,}")


# ------------------------------------------------------------
# 13. CURRENT MISSING VALUE SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("13. CURRENT MISSING VALUE SUMMARY")
print("=" * 100)

missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (
        df.isna().sum() / len(df) * 100
    )
})

missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values(
    by="Missing_Percentage",
    ascending=False
)

if missing_summary.empty:

    print("No missing values remain.")

else:

    print(missing_summary.to_string())


# ------------------------------------------------------------
# 14. MEMORY / DATASET INFORMATION
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("14. DATASET MEMORY USAGE")
print("=" * 100)

memory_usage_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)

print(f"Memory usage : {memory_usage_mb:.2f} MB")


# ------------------------------------------------------------
# FINAL
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("END OF CURRENT INPATIENT DATA AUDIT")
print("=" * 100)

CURRENT INPATIENT DATASET - COMPLETE DATA AUDIT

1. DATASET DIMENSIONS
Number of rows       : 40,474
Number of columns    : 28
Total cells         : 1,133,272

2. CURRENT COLUMN NAMES
01. BeneID
02. ClaimID
03. ClaimStartDt
04. ClaimEndDt
05. Provider
06. InscClaimAmtReimbursed
07. AttendingPhysician
08. OperatingPhysician
09. OtherPhysician
10. AdmissionDt
11. ClmAdmitDiagnosisCode
12. DischargeDt
13. DiagnosisGroupCode
14. ClmDiagnosisCode_1
15. ClmDiagnosisCode_2
16. ClmDiagnosisCode_3
17. ClmDiagnosisCode_4
18. ClmDiagnosisCode_5
19. ClmDiagnosisCode_6
20. ClmDiagnosisCode_7
21. ClmDiagnosisCode_8
22. ClmDiagnosisCode_9
23. ClmDiagnosisCode_10
24. ClmProcedureCode_1
25. ClmProcedureCode_2
26. ClmProcedureCode_3
27. ClmProcedureCode_4
28. ClmProcedureCode_5

3. COMPLETE COLUMN-WISE SUMMARY
                Column      Data_Type  Non_Null_Count  Missing_Count  Missing_Percentage  Unique_Count
                BeneID         string           40474              0            0.000000     

In [33]:
# ============================================================
# STEP 28: EMPTY / WHITESPACE VALUE CHECK
# ============================================================

print("=" * 80)
print("EMPTY AND WHITESPACE VALUE CHECK")
print("=" * 80)

text_columns = df.select_dtypes(include=["object", "string"]).columns

total_empty = 0

for col in text_columns:

    empty_count = (
        df[col]
        .astype("string")
        .str.strip()
        .eq("")
        .sum()
    )

    if empty_count > 0:
        print(f"{col:<30} : {empty_count:,} empty/whitespace values")
        total_empty += empty_count

if total_empty == 0:
    print("No empty or whitespace-only values found.")

print("=" * 80)

EMPTY AND WHITESPACE VALUE CHECK
No empty or whitespace-only values found.


In [34]:
# ============================================================
# STEP 29: CLAIM DURATION VALIDATION
# ============================================================

print("=" * 80)
print("CLAIM DURATION VALIDATION")
print("=" * 80)

claim_duration = (
    df["ClaimEndDt"] - df["ClaimStartDt"]
).dt.days

hospitalization_duration = (
    df["DischargeDt"] - df["AdmissionDt"]
).dt.days

print("\nCLAIM DURATION")
print("-" * 80)

print(f"Minimum duration : {claim_duration.min()} days")
print(f"Maximum duration : {claim_duration.max()} days")
print(f"Mean duration    : {claim_duration.mean():.2f} days")
print(f"Negative claims : {(claim_duration < 0).sum():,}")

print("\nHOSPITALIZATION DURATION")
print("-" * 80)

print(f"Minimum duration : {hospitalization_duration.min()} days")
print(f"Maximum duration : {hospitalization_duration.max()} days")
print(f"Mean duration    : {hospitalization_duration.mean():.2f} days")
print(
    f"Negative stays  : "
    f"{(hospitalization_duration < 0).sum():,}"
)

print("=" * 80)

CLAIM DURATION VALIDATION

CLAIM DURATION
--------------------------------------------------------------------------------
Minimum duration : 0 days
Maximum duration : 36 days
Mean duration    : 5.66 days
Negative claims : 0

HOSPITALIZATION DURATION
--------------------------------------------------------------------------------
Minimum duration : 0 days
Maximum duration : 35 days
Mean duration    : 5.67 days
Negative stays  : 0


In [35]:
# ============================================================
# STEP 30: ZERO REIMBURSEMENT CHECK
# ============================================================

print("=" * 80)
print("ZERO REIMBURSEMENT CHECK")
print("=" * 80)

zero_reimbursement = (
    df["InscClaimAmtReimbursed"] == 0
).sum()

zero_percentage = (
    zero_reimbursement / len(df) * 100
)

print(f"Zero reimbursement claims : {zero_reimbursement:,}")
print(f"Percentage                : {zero_percentage:.2f}%")

print("\nDecision:")
print("Zero reimbursement claims are retained.")
print("A zero financial value is not automatically considered an invalid claim.")

print("=" * 80)

ZERO REIMBURSEMENT CHECK
Zero reimbursement claims : 1,085
Percentage                : 2.68%

Decision:
Zero reimbursement claims are retained.
A zero financial value is not automatically considered an invalid claim.


In [36]:
# ============================================================
# STEP 31: FINAL MISSING VALUE CHECK
# ============================================================

print("=" * 80)
print("FINAL MISSING VALUE CHECK")
print("=" * 80)

missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (
        df.isna().sum() / len(df) * 100
    )
})

remaining_missing = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values(
    "Missing_Percentage",
    ascending=False
)

if remaining_missing.empty:

    print("No missing values remain.")

else:

    print("Remaining missing values:")
    print(remaining_missing.to_string())

print("=" * 80)

FINAL MISSING VALUE CHECK
Remaining missing values:
                    Missing_Count  Missing_Percentage
ClmProcedureCode_5          40465           99.977764
ClmProcedureCode_4          40358           99.713396
ClmProcedureCode_3          39509           97.615753
ClmProcedureCode_2          35020           86.524683
ClmProcedureCode_1          17326           42.807728


In [37]:
# ============================================================
# STEP 31: FINAL MISSING VALUE CHECK
# ============================================================

print("=" * 80)
print("FINAL MISSING VALUE CHECK")
print("=" * 80)

missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (
        df.isna().sum() / len(df) * 100
    )
})

remaining_missing = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values(
    "Missing_Percentage",
    ascending=False
)

if remaining_missing.empty:

    print("No missing values remain.")

else:

    print("Remaining missing values:")
    print(remaining_missing.to_string())

print("=" * 80)

FINAL MISSING VALUE CHECK
Remaining missing values:
                    Missing_Count  Missing_Percentage
ClmProcedureCode_5          40465           99.977764
ClmProcedureCode_4          40358           99.713396
ClmProcedureCode_3          39509           97.615753
ClmProcedureCode_2          35020           86.524683
ClmProcedureCode_1          17326           42.807728


In [38]:
# ============================================================
# STEP 32: FINAL DUPLICATE AND IDENTIFIER CHECK
# ============================================================

print("=" * 80)
print("FINAL DUPLICATE AND IDENTIFIER CHECK")
print("=" * 80)

print(f"Total rows              : {len(df):,}")
print(f"Duplicate complete rows : {df.duplicated().sum():,}")

print(f"\nUnique ClaimIDs         : {df['ClaimID'].nunique():,}")
print(
    f"Duplicate ClaimIDs     : "
    f"{df['ClaimID'].duplicated().sum():,}"
)

print(f"\nUnique BeneIDs          : {df['BeneID'].nunique():,}")
print(f"Unique Providers        : {df['Provider'].nunique():,}")

print("\nMissing identifiers:")

for col in ["ClaimID", "BeneID", "Provider"]:
    print(
        f"{col:<15}: "
        f"{df[col].isna().sum():,}"
    )

print("=" * 80)


FINAL DUPLICATE AND IDENTIFIER CHECK
Total rows              : 40,474
Duplicate complete rows : 0

Unique ClaimIDs         : 40,474
Duplicate ClaimIDs     : 0

Unique BeneIDs          : 31,289
Unique Providers        : 2,092

Missing identifiers:
ClaimID        : 0
BeneID         : 0
Provider       : 0


In [39]:
# ============================================================
# STEP 33: FINAL DATA TYPE CHECK
# ============================================================

print("=" * 80)
print("FINAL DATA TYPE CHECK")
print("=" * 80)

print(df.dtypes.to_string())

print("=" * 80)

FINAL DATA TYPE CHECK
BeneID                    string[python]
ClaimID                   string[python]
ClaimStartDt              datetime64[ns]
ClaimEndDt                datetime64[ns]
Provider                  string[python]
InscClaimAmtReimbursed             int64
AttendingPhysician        string[python]
OperatingPhysician        string[python]
OtherPhysician            string[python]
AdmissionDt               datetime64[ns]
ClmAdmitDiagnosisCode     string[python]
DischargeDt               datetime64[ns]
DiagnosisGroupCode        string[python]
ClmDiagnosisCode_1        string[python]
ClmDiagnosisCode_2        string[python]
ClmDiagnosisCode_3        string[python]
ClmDiagnosisCode_4        string[python]
ClmDiagnosisCode_5        string[python]
ClmDiagnosisCode_6        string[python]
ClmDiagnosisCode_7        string[python]
ClmDiagnosisCode_8        string[python]
ClmDiagnosisCode_9        string[python]
ClmDiagnosisCode_10       string[python]
ClmProcedureCode_1        string[py

In [40]:
# ============================================================
# STEP 34: FINAL INPATIENT PREPROCESSING SUMMARY
# ============================================================

print("=" * 80)
print("FINAL INPATIENT PREPROCESSING SUMMARY")
print("=" * 80)

print(f"Final rows                  : {df.shape[0]:,}")
print(f"Final columns               : {df.shape[1]:,}")
print(f"Duplicate rows              : {df.duplicated().sum():,}")
print(f"Duplicate ClaimIDs          : {df['ClaimID'].duplicated().sum():,}")
print(f"Unique beneficiaries        : {df['BeneID'].nunique():,}")
print(f"Unique providers            : {df['Provider'].nunique():,}")
print(f"Unique claims               : {df['ClaimID'].nunique():,}")

print(
    f"Negative reimbursements     : "
    f"{(df['InscClaimAmtReimbursed'] < 0).sum():,}"
)

print(
    f"Zero reimbursements         : "
    f"{(df['InscClaimAmtReimbursed'] == 0).sum():,}"
)

print(
    f"Missing values total        : "
    f"{df.isna().sum().sum():,}"
)

print("\nRemoved columns:")
print("- ClmProcedureCode_6 : completely empty")
print("- DeductibleAmtPaid  : zero variance")

print("\nMissing diagnosis/physician values:")
print("- Replaced with UNKNOWN")

print("\nFeature engineering:")
print("- NOT performed at this stage")

print("=" * 80)

FINAL INPATIENT PREPROCESSING SUMMARY
Final rows                  : 40,474
Final columns               : 28
Duplicate rows              : 0
Duplicate ClaimIDs          : 0
Unique beneficiaries        : 31,289
Unique providers            : 2,092
Unique claims               : 40,474
Negative reimbursements     : 0
Zero reimbursements         : 1,085
Missing values total        : 172,678

Removed columns:
- ClmProcedureCode_6 : completely empty
- DeductibleAmtPaid  : zero variance

Missing diagnosis/physician values:
- Replaced with UNKNOWN

Feature engineering:
- NOT performed at this stage


In [41]:
# ============================================================
# CHECK WHETHER TARGET VARIABLE IS AVAILABLE
# ============================================================

print("=" * 80)
print("TARGET VARIABLE CHECK")
print("=" * 80)

print("Current inpatient columns:")
print(df.columns.tolist())

print("\nPotentialFraud present:", "PotentialFraud" in df.columns)

print("=" * 80)

TARGET VARIABLE CHECK
Current inpatient columns:
['BeneID', 'ClaimID', 'ClaimStartDt', 'ClaimEndDt', 'Provider', 'InscClaimAmtReimbursed', 'AttendingPhysician', 'OperatingPhysician', 'OtherPhysician', 'AdmissionDt', 'ClmAdmitDiagnosisCode', 'DischargeDt', 'DiagnosisGroupCode', 'ClmDiagnosisCode_1', 'ClmDiagnosisCode_2', 'ClmDiagnosisCode_3', 'ClmDiagnosisCode_4', 'ClmDiagnosisCode_5', 'ClmDiagnosisCode_6', 'ClmDiagnosisCode_7', 'ClmDiagnosisCode_8', 'ClmDiagnosisCode_9', 'ClmDiagnosisCode_10', 'ClmProcedureCode_1', 'ClmProcedureCode_2', 'ClmProcedureCode_3', 'ClmProcedureCode_4', 'ClmProcedureCode_5']

PotentialFraud present: False


In [42]:
print("PotentialFraud" in df.columns)

False


In [43]:
print("PotentialFraud" in df.columns)

if "PotentialFraud" in df.columns:
    print(df[["Provider", "PotentialFraud"]].head())
    print(df["PotentialFraud"].value_counts())
else:
    print("PotentialFraud is NOT present in the inpatient dataframe.")

False
PotentialFraud is NOT present in the inpatient dataframe.


In [44]:
# ============================================================
# REMOVE HIGH-MISSING PROCEDURE COLUMNS
# ============================================================

print("=" * 80)
print("REMOVING PROCEDURE COLUMNS WITH >= 90% MISSING VALUES")
print("=" * 80)

# Based on the team lead's preprocessing guideline,
# procedure columns with 90% or more missing values
# are removed.

procedure_columns_to_remove = [
    "ClmProcedureCode_3",
    "ClmProcedureCode_4",
    "ClmProcedureCode_5"
]

# Display the columns before removal
print("\nColumns to be removed:")
for col in procedure_columns_to_remove:
    print(f"- {col}")

# Remove the selected columns
df.drop(
    columns=procedure_columns_to_remove,
    inplace=True
)

print("\nRemaining procedure columns:")

remaining_procedure_columns = [
    col for col in df.columns
    if col.startswith("ClmProcedureCode_")
]

for col in remaining_procedure_columns:
    print(f"- {col}")

print("\nCurrent dataset shape:")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]:,}")

print("=" * 80)

REMOVING PROCEDURE COLUMNS WITH >= 90% MISSING VALUES

Columns to be removed:
- ClmProcedureCode_3
- ClmProcedureCode_4
- ClmProcedureCode_5

Remaining procedure columns:
- ClmProcedureCode_1
- ClmProcedureCode_2

Current dataset shape:
Rows    : 40,474
Columns : 25


## 36. Verify Procedure Column Removal

Procedure-code columns with 90% or more missing values were removed according
to the preprocessing guideline.

This step verifies that the intended columns were removed and that the
retained procedure columns remain available for subsequent analysis and
feature engineering.

In [45]:
# ============================================================
# STEP 36: VERIFY PROCEDURE COLUMN REMOVAL
# ============================================================

print("=" * 80)
print("VERIFYING PROCEDURE COLUMN REMOVAL")
print("=" * 80)

removed_columns = [
    "ClmProcedureCode_3",
    "ClmProcedureCode_4",
    "ClmProcedureCode_5"
]

retained_columns = [
    "ClmProcedureCode_1",
    "ClmProcedureCode_2"
]

print("\nREMOVED COLUMNS")
print("-" * 80)

for col in removed_columns:
    print(f"{col:<25} : {'REMOVED' if col not in df.columns else 'STILL PRESENT'}")

print("\nRETAINED PROCEDURE COLUMNS")
print("-" * 80)

for col in retained_columns:
    print(f"{col:<25} : {'PRESENT' if col in df.columns else 'MISSING'}")

print("\nCURRENT DATASET SHAPE")
print("-" * 80)

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]:,}")

print("=" * 80)

VERIFYING PROCEDURE COLUMN REMOVAL

REMOVED COLUMNS
--------------------------------------------------------------------------------
ClmProcedureCode_3        : REMOVED
ClmProcedureCode_4        : REMOVED
ClmProcedureCode_5        : REMOVED

RETAINED PROCEDURE COLUMNS
--------------------------------------------------------------------------------
ClmProcedureCode_1        : PRESENT
ClmProcedureCode_2        : PRESENT

CURRENT DATASET SHAPE
--------------------------------------------------------------------------------
Rows    : 40,474
Columns : 25


In [46]:
# ============================================================
# STEP 37: RECHECK MISSING VALUES AFTER COLUMN REMOVAL
# ============================================================

print("=" * 80)
print("MISSING VALUE CHECK AFTER PROCEDURE COLUMN REMOVAL")
print("=" * 80)

missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (
        df.isna().sum() / len(df) * 100
    )
})

remaining_missing = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values(
    by="Missing_Percentage",
    ascending=False
)

if remaining_missing.empty:
    print("No missing values remain.")

else:
    print(remaining_missing.to_string())

print("=" * 80)

MISSING VALUE CHECK AFTER PROCEDURE COLUMN REMOVAL
                    Missing_Count  Missing_Percentage
ClmProcedureCode_2          35020           86.524683
ClmProcedureCode_1          17326           42.807728


In [47]:
# ============================================================
# FINAL DUPLICATE AND ID INTEGRITY CHECK
# ============================================================

print("=" * 80)
print("FINAL DUPLICATE AND ID INTEGRITY CHECK")
print("=" * 80)

print(f"Rows                  : {len(df):,}")
print(f"Duplicate rows        : {df.duplicated().sum():,}")
print(f"Unique ClaimIDs       : {df['ClaimID'].nunique():,}")
print(f"Duplicate ClaimIDs    : {df['ClaimID'].duplicated().sum():,}")
print(f"Unique BeneIDs        : {df['BeneID'].nunique():,}")
print(f"Unique Providers      : {df['Provider'].nunique():,}")

print("\nMissing identifiers:")

for col in ["ClaimID", "BeneID", "Provider"]:
    print(f"{col:<15}: {df[col].isna().sum():,}")

print("=" * 80)

FINAL DUPLICATE AND ID INTEGRITY CHECK
Rows                  : 40,474
Duplicate rows        : 0
Unique ClaimIDs       : 40,474
Duplicate ClaimIDs    : 0
Unique BeneIDs        : 31,289
Unique Providers      : 2,092

Missing identifiers:
ClaimID        : 0
BeneID         : 0
Provider       : 0


In [48]:
# ============================================================
# FINAL DATA TYPE CHECK
# ============================================================

print("=" * 80)
print("FINAL DATA TYPE CHECK")
print("=" * 80)

print(df.dtypes.to_string())

print("=" * 80)

FINAL DATA TYPE CHECK
BeneID                    string[python]
ClaimID                   string[python]
ClaimStartDt              datetime64[ns]
ClaimEndDt                datetime64[ns]
Provider                  string[python]
InscClaimAmtReimbursed             int64
AttendingPhysician        string[python]
OperatingPhysician        string[python]
OtherPhysician            string[python]
AdmissionDt               datetime64[ns]
ClmAdmitDiagnosisCode     string[python]
DischargeDt               datetime64[ns]
DiagnosisGroupCode        string[python]
ClmDiagnosisCode_1        string[python]
ClmDiagnosisCode_2        string[python]
ClmDiagnosisCode_3        string[python]
ClmDiagnosisCode_4        string[python]
ClmDiagnosisCode_5        string[python]
ClmDiagnosisCode_6        string[python]
ClmDiagnosisCode_7        string[python]
ClmDiagnosisCode_8        string[python]
ClmDiagnosisCode_9        string[python]
ClmDiagnosisCode_10       string[python]
ClmProcedureCode_1        string[py

##  Final Date Validation

A final chronological consistency check was performed after the inpatient
preprocessing operations.

The following date relationships were validated:

- `ClaimEndDt` should not occur before `ClaimStartDt`.
- `DischargeDt` should not occur before `AdmissionDt`.
- `DischargeDt` should not occur before `ClaimStartDt`.
- `AdmissionDt` should not occur after `ClaimEndDt`.
- `AdmissionDt` occurring before `ClaimStartDt` was separately identified
  for review.

The purpose of this validation was to identify impossible or potentially
inconsistent date relationships without automatically deleting records.

In [49]:
# ============================================================
# FINAL DATE VALIDATION
# ============================================================

print("=" * 80)
print("FINAL DATE VALIDATION")
print("=" * 80)

checks = {
    "ClaimEndDt < ClaimStartDt":
        (df["ClaimEndDt"] < df["ClaimStartDt"]).sum(),

    "DischargeDt < AdmissionDt":
        (df["DischargeDt"] < df["AdmissionDt"]).sum(),

    "DischargeDt < ClaimStartDt":
        (df["DischargeDt"] < df["ClaimStartDt"]).sum(),

    "AdmissionDt > ClaimEndDt":
        (df["AdmissionDt"] > df["ClaimEndDt"]).sum(),

    "AdmissionDt < ClaimStartDt":
        (df["AdmissionDt"] < df["ClaimStartDt"]).sum()
}

for condition, count in checks.items():
    print(f"{condition:<35}: {count:,}")

print("=" * 80)

FINAL DATE VALIDATION
ClaimEndDt < ClaimStartDt          : 0
DischargeDt < AdmissionDt          : 0
DischargeDt < ClaimStartDt         : 0
AdmissionDt > ClaimEndDt           : 0
AdmissionDt < ClaimStartDt         : 32


### Observation

No violations were identified for the following chronological checks:

- Claim end date before claim start date
- Discharge date before admission date
- Discharge date before claim start date
- Admission date after claim end date

A total of 32 records were identified where `AdmissionDt` occurred before
`ClaimStartDt`.

These records were retained because this relationship alone was not treated
as sufficient evidence that the claims were invalid. The observation was
documented for transparency rather than removing potentially valid claims.

Therefore, no records were removed based on the final date validation.

## Final Inpatient Dataset Shape

After completing the major preprocessing and column-removal operations, the
final dimensions and column structure of the inpatient dataset were
validated.

This step establishes the final claim-level structure before the cleaned
dataset is handed over for team-level integration.

The final dataset contains 40,474 inpatient claim records and 25 columns.

The dataset retains the following major information categories:

- Beneficiary identifier
- Claim identifier
- Provider identifier
- Claim dates
- Admission and discharge dates
- Reimbursement amount
- Physician identifiers
- Admission diagnosis
- Diagnosis group
- Diagnosis codes
- Retained procedure codes

The procedure fields retained at this stage are:

- `ClmProcedureCode_1`
- `ClmProcedureCode_2`

Procedure fields with 90% or more missingness were removed according to the
team's preprocessing guideline.

The dataset remains at claim level and no provider-level aggregation or
feature engineering has been applied.

In [50]:
print("=" * 80)
print("FINAL INPATIENT DATASET SHAPE")
print("=" * 80)

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]:,}")

print("\nColumns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:02d}. {col}")

print("=" * 80)

FINAL INPATIENT DATASET SHAPE
Rows    : 40,474
Columns : 25

Columns:
01. BeneID
02. ClaimID
03. ClaimStartDt
04. ClaimEndDt
05. Provider
06. InscClaimAmtReimbursed
07. AttendingPhysician
08. OperatingPhysician
09. OtherPhysician
10. AdmissionDt
11. ClmAdmitDiagnosisCode
12. DischargeDt
13. DiagnosisGroupCode
14. ClmDiagnosisCode_1
15. ClmDiagnosisCode_2
16. ClmDiagnosisCode_3
17. ClmDiagnosisCode_4
18. ClmDiagnosisCode_5
19. ClmDiagnosisCode_6
20. ClmDiagnosisCode_7
21. ClmDiagnosisCode_8
22. ClmDiagnosisCode_9
23. ClmDiagnosisCode_10
24. ClmProcedureCode_1
25. ClmProcedureCode_2


### Observation

The final inpatient preprocessing dataframe contains:

- 40,474 rows
- 25 columns

The dataset remains at the original claim-level grain.

`ClmProcedureCode_1` and `ClmProcedureCode_2` were retained because their
missingness was below the team-defined 90% threshold.

`ClmProcedureCode_3`, `ClmProcedureCode_4`, `ClmProcedureCode_5`, and
`ClmProcedureCode_6` were removed because they met or exceeded the
90% missingness threshold.

`DeductibleAmtPaid` was also removed earlier because its observed values
were constant at 1068, resulting in zero variance.

No provider-level features, fraud scores, or `PotentialFraud` target values
were added to this independent inpatient preprocessing dataset.

The resulting dataframe is therefore ready for final quality checks and
handoff to the team for the subsequent unified-dataset and feature-
engineering stage.

In [51]:
# ============================================================
# STEP 32: SAVE FINAL PREPROCESSED INPATIENT DATASET
# ============================================================

print("=" * 80)
print("SAVING FINAL PREPROCESSED INPATIENT DATASET")
print("=" * 80)

# Define output file path
output_path = "/kaggle/working/cleaned_inpatient.csv"

# Save the final dataframe
df.to_csv(output_path, index=False)

print("\nFILE SAVED SUCCESSFULLY")
print("-" * 80)
print(f"File path : {output_path}")
print(f"Rows      : {df.shape[0]:,}")
print(f"Columns   : {df.shape[1]:,}")

print("\nFinal procedure columns:")
for col in df.columns:
    if col.startswith("ClmProcedureCode"):
        print(f"- {col}")

print("\nFinal dataset verification:")
print(f"File exists : {os.path.exists(output_path)}")

print("=" * 80)

SAVING FINAL PREPROCESSED INPATIENT DATASET

FILE SAVED SUCCESSFULLY
--------------------------------------------------------------------------------
File path : /kaggle/working/cleaned_inpatient.csv
Rows      : 40,474
Columns   : 25

Final procedure columns:
- ClmProcedureCode_1
- ClmProcedureCode_2

Final dataset verification:
File exists : True


In [52]:
# ============================================================
# STEP 33: VERIFY SAVED CSV
# ============================================================

print("=" * 80)
print("VERIFYING SAVED PREPROCESSED CSV")
print("=" * 80)

saved_df = pd.read_csv(output_path)

print(f"Saved rows    : {saved_df.shape[0]:,}")
print(f"Saved columns : {saved_df.shape[1]:,}")

print("\nShape comparison:")
print(f"Original dataframe : {df.shape}")
print(f"Saved CSV          : {saved_df.shape}")

print("\nColumn consistency:")
print("Columns identical :", list(df.columns) == list(saved_df.columns))

print("=" * 80)

VERIFYING SAVED PREPROCESSED CSV
Saved rows    : 40,474
Saved columns : 25

Shape comparison:
Original dataframe : (40474, 25)
Saved CSV          : (40474, 25)

Column consistency:
Columns identical : True
